In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Narela, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,154.99,254.17,24.73,21.98,31.78,36.84,4.45,1.02,3.09,7.18,278.30,86.15,0.42,13.31,981.39,13.24,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,171.64,274.26,18.74,22.27,27.08,37.16,8.11,1.14,3.24,7.98,207.66,87.03,0.41,13.35,979.42,13.29,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,190.16,308.72,34.80,23.30,40.66,40.50,9.96,1.37,3.34,11.89,286.27,89.71,0.33,13.57,979.79,14.27,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,177.17,261.52,31.05,24.39,38.21,42.80,8.36,1.37,3.83,13.45,306.37,92.30,0.90,13.68,979.78,13.95,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,193.95,330.59,33.98,26.32,41.62,42.95,7.85,1.18,3.82,10.93,128.19,85.55,0.96,13.88,979.23,14.28,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,390.00,544.22,8.44,34.44,25.19,66.86,25.32,1.38,66.67,1.76,110.05,57.28,0.46,228.22,997.58,18.28,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,308.96,451.41,9.08,37.76,27.46,65.53,24.92,1.31,60.10,1.72,82.70,59.41,0.38,218.12,997.88,17.97,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,266.52,391.75,7.36,33.67,23.92,63.58,22.15,1.28,51.89,1.50,66.05,58.72,0.38,214.07,997.54,17.96,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,306.82,436.43,9.80,38.72,28.61,65.25,20.90,1.14,51.93,1.57,90.13,58.09,0.39,198.21,997.93,17.96,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
          From Date           To Date   PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  154.99  254.17  24.73  21.98  31.78   
1  02-01-2025 00:00  03-01-2025 00:00  171.64  274.26  18.74  22.27  27.08   
2  03-01-2025 00:00  04-01-2025 00:00  190.16  308.72   9.84  23.30  40.66   
3  04-01-2025 00:00  05-01-2025 00:00  177.17  261.52  31.05  24.39  38.21   
4  05-01-2025 00:00  06-01-2025 00:00  193.95  330.59   9.84  26.32  41.62   

     NH3   SO2    CO  Ozone  Benzene  Toluene     RH    WS     WD      BP  \
0  36.84  4.45  1.02   3.09     1.92   278.30  86.15  0.42  13.31  981.39   
1  37.16  8.11  1.14   3.24     1.92   207.66  87.03  0.41  13.35  979.42   
2  40.50  9.96  1.37   3.34     1.92   286.27  89.71  0.33  13.57  979.79   
3  42.80  8.36  1.37   3.83     1.92   306.37  92.30  0.90  13.68  979.78   
4  42.95  7.85  1.18   3.82     1.92   128.19  85.55  0.96  13.88  979.23   

      AT   RF  TOT-RF  
0  13.24  0.0     0.0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,1.437913,0.722118,2.196221,0.066611,1.483332,0.609380,-0.902800,0.296119,-1.241653,-0.188461,1.873921,1.724505,-0.813382,-1.899309,0.434226,-2.089784,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,1.758392,0.923108,1.197076,0.099101,0.840944,0.645775,-0.073199,0.872741,-1.234719,-0.188461,0.906031,1.797819,-0.832134,-1.898874,0.345719,-2.081739,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,2.114864,1.267863,-0.287463,0.214497,2.697034,1.025654,0.346134,1.977933,-1.230097,-0.188461,1.983124,2.021092,-0.982147,-1.896482,0.362342,-1.924052,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,1.864833,0.795651,3.250410,0.336614,2.362172,1.287246,-0.016533,1.977933,-1.207449,-0.188461,2.258528,2.236868,0.086697,-1.895286,0.361893,-1.975541,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.187814,1.486661,-0.287463,0.552840,2.828245,1.304307,-0.132133,1.064948,-1.207911,-0.188461,-0.182844,1.674519,0.199207,-1.893111,0.337182,-1.922443,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.189503,-0.139369,-0.520986,1.462558,0.582623,-0.025836,-0.154799,2.025984,1.697066,-0.290916,-0.431394,-0.680682,-0.738376,0.437707,1.161606,-1.278823,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.189503,2.695403,-0.414232,1.834511,0.892882,-0.025836,-0.154799,1.689622,1.393395,-0.316530,-0.806136,-0.503229,-0.888389,0.327876,1.175084,-1.328703,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.189503,2.098535,-0.701132,1.376291,0.409041,-0.025836,-0.154799,1.545466,1.013923,-0.457406,-1.034269,-0.560714,-0.888389,0.283835,1.159808,-1.330312,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.189503,2.545536,-0.294135,1.942064,1.050062,-0.025836,2.825870,0.872741,1.015771,-0.412582,-0.704332,-0.613200,-0.869637,0.111367,1.177330,-1.330312,0.0,0.0


In [10]:
df.to_excel('Narela2025.xlsx', index=False)